The IMDB dataset is a collection of movie reviews (text) and a label, `1` means the reviewer likes the movie and `0` means they did not like it.

# Part I (FastText Classification)

## Data preprocessing

In [17]:
import fasttext
from keras.datasets import imdb
from keras.preprocessing.sequence import pad_sequences
import os
import numpy as np
from sklearn.metrics import accuracy_score
import os
import numpy as np
import itertools
from sklearn.metrics import accuracy_score
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns



In [18]:
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=10000)
print(f"Training data shape: {x_train.shape}, {y_train.shape}")
print(f"Test data shape: {x_test.shape}, {y_test.shape}")
word_index = imdb.get_word_index()
rev_index = {v+1: k for k, v in word_index.items()}
rev_index.update({0:"<PAD>"})

def decode(x): return ' '.join(rev_index.get(i, '?') for i in x)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


/home/prnamhr/projects/Machine Learning/venv/lib/python3.12/site-packages/numpy/lib/_format_impl.py:838: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  array = pickle.load(fp, **pickle_kwargs)


Training data shape: (25000,), (25000,)
Test data shape: (25000,), (25000,)
1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Create two files: `imdb_train.txt` and `imdb_test.txt`.

* `imdb_train.txt` should contain the first 10,000 examples.

* `imdb_test.txt` should contain the first 5,000 examples.

Each line in these files should follow the format:

`f"__label__{y} {decode(x)}"`

Where:

* `y` is the label (e.g., 0 or 1),

* `x` is the input (e.g., a sequence of token IDs or bytes),

`decode(x)` converts `x` into a readable text string.
To learn more about FastText Embedding please see: https://fasttext.cc/docs/en/supervised-tutorial.html.

In [19]:
with open('imdb_train.txt', 'w', encoding='utf-8') as f:
    for i in range(min(10000, len(x_train))):
        decoded_text = decode(x_train[i])
        cleaned_text = ' '.join(decoded_text.split())
        f.write(f"__label__{y_train[i]} {cleaned_text}\n")

print(f"Created imdb_train.txt with {min(10000, len(x_train))} examples")


with open('imdb_test.txt', 'w', encoding='utf-8') as f:
    for i in range(min(5000, len(x_test))):
        decoded_text = decode(x_test[i])
        cleaned_text = ' '.join(decoded_text.split())
        f.write(f"__label__{y_test[i]} {cleaned_text}\n")

print(f"Created imdb_test.txt with {min(5000, len(x_test))} examples")

Created imdb_train.txt with 10000 examples
Created imdb_test.txt with 5000 examples


Now, train a model that learns the word2vec embeddings of the data, averages them for each review, and classifies the review as either 0 or 1. You can use the function `fasttext.train_supervised("imdb_train.txt")` for this purpose.

The train_supervised function accepts several parameters:

* `lr: Learning rate`
* `epoch: Number of training iterations`
* `wordNgrams: Maximum length of word n-gram`
* `dim: Size of word vectors`

First, try training the model using the default parameters. Then, experiment by tweaking these parameters to improve performance. Compare the results and see which configuration yields better accuracy.


In [20]:
model_default = fasttext.train_supervised("imdb_train.txt")
result_default = model_default.test("imdb_test.txt")
print(f"Default Parameters - Precision: {result_default[1]:.4f}")

Read 2M words
Number of words:  9996
Number of labels: 2
Progress:  57.9% words/sec/thread: 2033411 lr:  0.042064 avg.loss:  0.644025 ETA:   0h 0m 0s

Default Parameters - Precision: 0.8084


Progress: 100.0% words/sec/thread: 1840927 lr:  0.000000 avg.loss:  0.577391 ETA:   0h 0m 0s


In [ ]:
fasttext_params = {
    'lr': [0.1, 0.2, 0.5],
    'epoch': [10, 20, 30, 50],
    'wordNgrams': [1, 2, 3],
    'dim': [100, 150, 200]
}

best_precision = 0
best_params = {}
results_fasttext = []

for lr in fasttext_params['lr']:
    for epoch in fasttext_params['epoch']:
        for wordNgrams in fasttext_params['wordNgrams']:
            for dim in fasttext_params['dim']:
                try:
                    print(f"Testing: lr={lr}, epoch={epoch}, wordNgrams={wordNgrams}, dim={dim}")
                    model = fasttext.train_supervised("imdb_train.txt", lr=lr, epoch=epoch,wordNgrams=wordNgrams, dim=dim, verbose=0)
                    results = model.test("imdb_test.txt")
                    precision = results[1]

                    results_fasttext.append({
                        'lr': lr, 'epoch': epoch, 'wordNgrams': wordNgrams,
                        'dim': dim, 'precision': precision
                    })

                    if precision > best_precision:
                        best_precision = precision
                        best_params = {'lr': lr, 'epoch': epoch, 'wordNgrams': wordNgrams, 'dim': dim}
                    print(f"  Precision: {precision:.4f}")
                except Exception as e:
                    print(f"  Error: {e}")
                    continue

print(f"\nBest FastText Parameters: {best_params}")
print(f"Best Precision: {best_precision:.4f}")

Testing: lr=0.1, epoch=10, wordNgrams=1, dim=100
  Precision: 0.8596
Testing: lr=0.1, epoch=10, wordNgrams=1, dim=150
  Precision: 0.8600
Testing: lr=0.1, epoch=10, wordNgrams=1, dim=200
  Precision: 0.8612
Testing: lr=0.1, epoch=10, wordNgrams=2, dim=100
  Precision: 0.8344
Testing: lr=0.1, epoch=10, wordNgrams=2, dim=150
  Precision: 0.8318
Testing: lr=0.1, epoch=10, wordNgrams=2, dim=200
  Precision: 0.8320
Testing: lr=0.1, epoch=10, wordNgrams=3, dim=100
  Precision: 0.7562
Testing: lr=0.1, epoch=10, wordNgrams=3, dim=150
  Precision: 0.7452
Testing: lr=0.1, epoch=10, wordNgrams=3, dim=200
  Precision: 0.7426
Testing: lr=0.1, epoch=20, wordNgrams=1, dim=100
  Precision: 0.8708
Testing: lr=0.1, epoch=20, wordNgrams=1, dim=150
  Precision: 0.8700
Testing: lr=0.1, epoch=20, wordNgrams=1, dim=200
  Precision: 0.8702
Testing: lr=0.1, epoch=20, wordNgrams=2, dim=100
  Precision: 0.8766
Testing: lr=0.1, epoch=20, wordNgrams=2, dim=150
  Precision: 0.8760
Testing: lr=0.1, epoch=20, wordNgr

# Part II (Classification with LSTM)

## Data preprocessing

In [ ]:
from keras.datasets import imdb
from keras.models import Sequential
from keras.layers import Embedding, LSTM, Dense
from keras.preprocessing.sequence import pad_sequences

x_train_lstm, x_test_lstm = pad_sequences(x_train, maxlen=200), pad_sequences(x_test, maxlen=200)

Create an `LSTM model` where the first layer is an `Embedding layer` with a vocabulary size of `200`, and each word is mapped to a `128-dimensional` vector. The second layer should be an LSTM layer with `64 units`, followed by a final Dense layer with a `sigmoid activation function` for binary classification. Compile the model using the `binary_crossentropy` loss function, the `adam` optimizer, and `acc` as the evaluation metric. Train the model for three `epochs` with a batch size of `64`. What happens when you change the dimension of the `Embedding layer` and the number of `epochs`?

In [ ]:
def create_lstm_model(embedding_dim=128, lstm_units=64, vocab_size=10000, sequence_length=200):
    model = Sequential()
    model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=sequence_length))
    model.add(LSTM(units=lstm_units))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

# Train the model with default parameters
model_lstm_default = create_lstm_model()
history = model_lstm_default .fit(x_train_lstm, y_train, epochs=3, batch_size=64, validation_data=(x_test_lstm, y_test), verbose=1)

# Evaluate the model
loss, accuracy = model_lstm_default .evaluate(x_test_lstm, y_test, verbose=0)
print(f"LSTM Model Accuracy: {accuracy:.4f}")

Epoch 1/3


C:\Users\parna\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


391/391 ━━━━━━━━━━━━━━━━━━━━ 21s 50ms/step - accuracy: 0.8131 - loss: 0.4023 - val_accuracy: 0.8483 - val_loss: 0.3578
Epoch 2/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 19s 49ms/step - accuracy: 0.9094 - loss: 0.2364 - val_accuracy: 0.8578 - val_loss: 0.3308
Epoch 3/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 19s 49ms/step - accuracy: 0.9248 - loss: 0.1970 - val_accuracy: 0.8632 - val_loss: 0.3837
LSTM Model Accuracy: 0.8632


In [ ]:
embedding_dims = [64, 128, 256]

epoch_counts = [3, 5, 10]

best_accuracy = 0
best_params = {}
results_lstm = []

for emb_dim in embedding_dims:
    for epochs in epoch_counts:
        print(f"\nTraining LSTM with embedding_dim={emb_dim}, epochs={epochs}")

        model_lstm = create_lstm_model(embedding_dim=emb_dim)
        model_lstm.fit(x_train_lstm, y_train, epochs=epochs, batch_size=64, validation_data=(x_test_lstm, y_test),verbose=0)

        loss, accuracy = model_lstm.evaluate(x_test_lstm, y_test, verbose=0)
        print(f"Test Accuracy: {accuracy:.4f}")

        results_lstm.append({
            'embedding_dim': emb_dim,
            'epochs': epochs,
            'accuracy': accuracy
        })

        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_params = {'embedding_dim': emb_dim, 'epochs': epochs}

print(f"\nBest Accuracy: {best_accuracy:.4f}")
print(f"Best Parameters: {best_params}")



Training LSTM with embedding_dim=64, epochs=3
Test Accuracy: 0.8503

Training LSTM with embedding_dim=64, epochs=5
Test Accuracy: 0.8307

Training LSTM with embedding_dim=64, epochs=10
Test Accuracy: 0.8492

Training LSTM with embedding_dim=128, epochs=3
Test Accuracy: 0.8560

Training LSTM with embedding_dim=128, epochs=5
Test Accuracy: 0.8569

Training LSTM with embedding_dim=128, epochs=10
Test Accuracy: 0.8491

Training LSTM with embedding_dim=256, epochs=3
Test Accuracy: 0.8537

Training LSTM with embedding_dim=256, epochs=5
Test Accuracy: 0.8481

Training LSTM with embedding_dim=256, epochs=10
Test Accuracy: 0.8438

Best Accuracy: 0.8569
Best Parameters: {'embedding_dim': 128, 'epochs': 5}


# Part II (Classification with BERT)

## Data preprocessing

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch

dataset = load_dataset("imdb", split="train[:100000]+test[:5000]").train_test_split(test_size=0.2)
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch): return tokenizer(batch["text"], truncation=True, padding=True)
dataset = dataset.map(tokenize, batched=True)
dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

Map: 100%|██████████| 6000/6000 [00:01<00:00, 5456.90 examples/s]


* Upload a bert model called `distilbert-base-uncased` with `AutoModelForSequenceClassification`, make sure to set `num_labels` to `2`. Because our sentiment analysis has two classes.
* Set the training arguments using `TrainingArguments`, make sure to set `num_train_epochs` and `output_dir` and set `report_to="none"`.
* Create a `Trainer` and set its `model` and `args` to what you defined in th previous steps. Make sure to set `train_dataset=dataset["train"]`, and `eval_dataset=dataset["test"]`.
* Call the trainer by `.train()`
* Once the model is trained compute its accuracy using the following method (what happens when you increase `num_train_epochs`):

In [ ]:
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {"accuracy": accuracy_score(labels, preds)}

epoch_configs = [1, 3]
results_bert = []

for num_epochs in epoch_configs:
    print(f"\nTraining BERT with {num_epochs} epochs...")

    model_bert = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

    # Set training arguments
    training_args = TrainingArguments(
        output_dir=f'./bert_results_{num_epochs}',
        num_train_epochs=num_epochs,
        report_to="none"
    )

    # Create trainer
    trainer = Trainer(
        model=model_bert,
        args=training_args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["test"],
        compute_metrics=compute_metrics
    )

    # Train the model
    trainer.train()

    # Evaluate
    eval_results = trainer.evaluate()
    accuracy = eval_results['eval_accuracy']

    results_bert.append({
        'epochs': num_epochs,
        'accuracy': accuracy,
        'eval_loss': eval_results['eval_loss']
    })

    print(f"BERT ({num_epochs} epochs) Accuracy: {accuracy:.4f}")

# Display results
results_df = pd.DataFrame(results)
print("\nBERT Results Summary:")
print(results_df.to_string(index=False))



Training BERT with 1 epochs...


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\parna\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
500,0.383700
1000,0.332500
1500,0.285000
2000,0.281200
2500,0.256000
3000,0.268600


C:\Users\parna\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERT (1 epochs) Accuracy: 0.9237

Training BERT with 3 epochs...


C:\Users\parna\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
500,0.380200
1000,0.334800
1500,0.310900
2000,0.292900
2500,0.284500
3000,0.275100
3500,0.174600
4000,0.180400
4500,0.168500


C:\Users\parna\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [ ]:
print("\n=== SUMMARY OF ALL MODELS ===")
print(f"FastText Best Precision: {best_precision:.4f}")
print(f"LSTM Best Accuracy: {best_accuracy:.4f}")

# Create summary DataFrames
fasttext_df = pd.DataFrame(results_fasttext)
lstm_df = pd.DataFrame(results_lstm)

print("\nFastText Results (top 5):")
print(fasttext_df.nlargest(5, 'precision').to_string(index=False))

print("\nLSTM Results:")
print(lstm_df.to_string(index=False))

# Clean up temporary files
try:
    os.remove('imdb_train.txt')
    os.remove('imdb_test.txt')
    print("\nCleaned up temporary files.")
except:
    pass


=== SUMMARY OF ALL MODELS ===


NameError: name 'best_precision' is not defined